In [1]:

#* importaciones necesarias
from pathlib import Path

#* Cargador de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

#* División de texto en fragmentos 
from langchain_text_splitters import RecursiveCharacterTextSplitter

#* Modelos de Ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings

#* Vector store basado en Chroma
from langchain_chroma import Chroma

#* Prompting y construcción de la cadena RAG 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [2]:
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "assets"

PDF_NESTLE_PATH = DATA_DIR / "basesycondicionesnestle.pdf"

In [3]:

#* Creamos el loader para PDF
loader = PyPDFLoader(str(PDF_NESTLE_PATH))

documents = loader.load()

print(f"Número de páginas cargadas desde el PDF: {len(documents)}")
print("Ejemplo de contenido (primeros 500 caracteres de la primera página):\n")
print(documents[0].page_content[:500], "...")

Número de páginas cargadas desde el PDF: 8
Ejemplo de contenido (primeros 500 caracteres de la primera página):

BASES Y MECÁNICAS DE LA PROMOCIÓN 
 
“SU PARTICIPACIÓN EN LA PRESENTE PROMOCIÓN COMERCIAL, CONSTITUYE SU 
ADHESIÓN Y ACEPTACIÓN SIN RESERVA ALGUNA DE LOS PRESENTES TÉRMINOS, 
CONDICIONES, RESTRICCIONES Y AVISO DE PRIVACIDAD, POR LO QUE LE 
RECOMENDAMOS QUE ANTES DE PARTICIPAR LAS LEA CUIDADOSAMENTE A FIN DE QUE 
LAS ANALICE Y CONCIENTEMENTE PARTICIPE O SE ABSTENGA DE ELLO. 
 
Nombre de la promoción: “Ofertas de Hoy con Inés Básicos” 
 
Responsable y organizadora de la promoción: Desarrollo Comer ...


In [4]:

#* Dividimos el texto en fragmentos manejables
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, #* Tamaño aproximado de cada fragmento en caracteres
    chunk_overlap=200, #* Superposición entre fragmentos para mantener contexto
    add_start_index=True 
)

#* Aplicamos el splitter a la lista de documentos cargados
splits = text_splitter.split_documents(documents)

print(f"Número de fragmentos creados: {len(splits)}")
print("Ejemplo de fragmento:\n")
print(splits[0].page_content[:500], "...")


Número de fragmentos creados: 35
Ejemplo de fragmento:

BASES Y MECÁNICAS DE LA PROMOCIÓN 
 
“SU PARTICIPACIÓN EN LA PRESENTE PROMOCIÓN COMERCIAL, CONSTITUYE SU 
ADHESIÓN Y ACEPTACIÓN SIN RESERVA ALGUNA DE LOS PRESENTES TÉRMINOS, 
CONDICIONES, RESTRICCIONES Y AVISO DE PRIVACIDAD, POR LO QUE LE 
RECOMENDAMOS QUE ANTES DE PARTICIPAR LAS LEA CUIDADOSAMENTE A FIN DE QUE 
LAS ANALICE Y CONCIENTEMENTE PARTICIPE O SE ABSTENGA DE ELLO. 
 
Nombre de la promoción: “Ofertas de Hoy con Inés Básicos” 
 
Responsable y organizadora de la promoción: Desarrollo Comer ...


In [5]:

#* Modelo de embeddings basado en HuggingFace
model_embedding = OllamaEmbeddings(model="embeddinggemma:300m")

print("Objeto de embeddings creado correctamente.")

Objeto de embeddings creado correctamente.


In [6]:

#* Directorio donde se va almacenar la base vectorial
CHROMA_DIR = BASE_DIR / "chroma_db"
persist_dir = str(CHROMA_DIR / "db_nestle_langchain")

#* Creamos el vector store vacío
vector_store = Chroma(
    collection_name="nestle_collection",
    embedding_function=model_embedding,
    persist_directory=persist_dir
)

#* Indexamos (añadimos) todos los fragmentos al vector store
documents_ids = vector_store.add_documents(splits)

print(f"Se indexaron {len(documents_ids)} fragmentos en Chroma.")
print(f"La base vectorial se guardó en la carpeta: {persist_dir}")

Se indexaron 35 fragmentos en Chroma.
La base vectorial se guardó en la carpeta: d:\nalvarez\100_cursos\bases_vectoriales\chroma_db\db_nestle_langchain


In [7]:
#* Creación de retriever a partir del vector store
retriever_todo_claro = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever creado correctamente a partir del vector store.")

Retriever creado correctamente a partir del vector store.


In [8]:
#* Cargamos el modelo de Ollama a usar
model_llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.3
)

#* Prompt siguiendo la filosofía RAG
template = """
Eres un asistente especializado en términos y condiciones de promociones de consumo masivo.

Uso EXCLUSIVAMENTE la información del contexto para responder en español latinoamericano y en un máximo de 4 párrafos. Si la pregunta no se puede responder con el contexto, di claramente que la información no esta en el documento.

Pregunta del usuario: 
{question}

Contexto:
{context}
"""

prompt = ChatPromptTemplate.from_template(template)

print("Modelo de chat y prompt RAG configurados.")

Modelo de chat y prompt RAG configurados.


In [9]:

#* Creamos la cadena RAG

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever_todo_claro,
    }
    | prompt
    | model_llm
)

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [10]:
#* Probamos la cadena RAG con una pregunta de ejemplo

pregunta = (
    "Según las bases de la promoción <<Ofertas de Hoy con Inés "
    "básico>>, ¿en qué situaciones un participante queda descalificado o "
    "se le anula su participación en la promoción?"
)

respuesta = rag_chain.invoke(pregunta)

print("Pregunta:")
print(pregunta)
print("\nRespuesta generada por el modelo:")
print(respuesta.content)

Pregunta:
Según las bases de la promoción <<Ofertas de Hoy con Inés básico>>, ¿en qué situaciones un participante queda descalificado o se le anula su participación en la promoción?

Respuesta generada por el modelo:
Según las bases de la promoción “Ofertas de Hoy con Inés Básicos”, un participante queda descalificado o se le anula su participación en la promoción si registra datos incorrectos en su ticket de compra o en la urna de participación, incluyendo tachaduras o sobre escrituras. Además, si el participante no calcula la cantidad exacta de imágenes de “Inés” o que exceda la cantidad correcta, también se anula su participación.



In [11]:

#* Visualizamos las fuentes utilizadas para responder
print("Pregunta de prueba:")
print(pregunta)
print("\nFragmentos recuperados:\n")
docs_relevantes = retriever_todo_claro.invoke(pregunta)
print(f"Se recuperaron {len(docs_relevantes)} fragmentos relevantes:\n")
for i, doc in enumerate(docs_relevantes, 1):
    print(f"Fragmento {i}:\n{doc.page_content}\n{'-'*50}\n")

Pregunta de prueba:
Según las bases de la promoción <<Ofertas de Hoy con Inés básico>>, ¿en qué situaciones un participante queda descalificado o se le anula su participación en la promoción?

Fragmentos recuperados:

Se recuperaron 4 fragmentos relevantes:

Fragmento 1:
BASES Y MECÁNICAS DE LA PROMOCIÓN 
 
“SU PARTICIPACIÓN EN LA PRESENTE PROMOCIÓN COMERCIAL, CONSTITUYE SU 
ADHESIÓN Y ACEPTACIÓN SIN RESERVA ALGUNA DE LOS PRESENTES TÉRMINOS, 
CONDICIONES, RESTRICCIONES Y AVISO DE PRIVACIDAD, POR LO QUE LE 
RECOMENDAMOS QUE ANTES DE PARTICIPAR LAS LEA CUIDADOSAMENTE A FIN DE QUE 
LAS ANALICE Y CONCIENTEMENTE PARTICIPE O SE ABSTENGA DE ELLO. 
 
Nombre de la promoción: “Ofertas de Hoy con Inés Básicos” 
 
Responsable y organizadora de la promoción: Desarrollo Comercial Abarrotero SA de CV. Con 
domicilio carr. cortazar-estacion km 1.5 s/n predio Santa Anita c.p 38300 Cortazar, Gto 
 
 
Cobertura Geográfica: Nacional, a través de todas las tiendas Básicos (ver anexo A para identificar 
las

### Experimento 1

In [12]:

#* Chunks más pequeños, menos fragmentos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350, 
    chunk_overlap=20, 
    add_start_index=True 
)

splits = text_splitter.split_documents(documents)

print(f"Número de fragmentos creados: {len(splits)}")
print("Ejemplo de fragmento:\n")
print(splits[0].page_content[:500], "...")


Número de fragmentos creados: 90
Ejemplo de fragmento:

BASES Y MECÁNICAS DE LA PROMOCIÓN 
 
“SU PARTICIPACIÓN EN LA PRESENTE PROMOCIÓN COMERCIAL, CONSTITUYE SU 
ADHESIÓN Y ACEPTACIÓN SIN RESERVA ALGUNA DE LOS PRESENTES TÉRMINOS, 
CONDICIONES, RESTRICCIONES Y AVISO DE PRIVACIDAD, POR LO QUE LE 
RECOMENDAMOS QUE ANTES DE PARTICIPAR LAS LEA CUIDADOSAMENTE A FIN DE QUE ...


In [13]:

#* Directorio donde se va almacenar la base vectorial
CHROMA_DIR = BASE_DIR / "chroma_db"
persist_dir = str(CHROMA_DIR / "db_nestle_v1_langchain")

#* Creamos el vector store vacío
vector_store = Chroma(
    collection_name="nestle_v1_collection",
    embedding_function=model_embedding,
    persist_directory=persist_dir
)

#* Indexamos (añadimos) todos los fragmentos al vector store
documents_ids = vector_store.add_documents(splits)

print(f"Se indexaron {len(documents_ids)} fragmentos en Chroma.")
print(f"La base vectorial se guardó en la carpeta: {persist_dir}")

Se indexaron 90 fragmentos en Chroma.
La base vectorial se guardó en la carpeta: d:\nalvarez\100_cursos\bases_vectoriales\chroma_db\db_nestle_v1_langchain


In [14]:
#* Creación de retriever a partir del vector store
retriever_nestle_v1 = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever creado correctamente a partir del vector store.")

Retriever creado correctamente a partir del vector store.


In [15]:
rag_chain_v1 = (
    {
        "question": RunnablePassthrough(),
        "context": retriever_nestle_v1,
    }
    | prompt
    | model_llm
)

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [16]:
pregunta = (
    "Según los términos y condiciones del beneficio Todo Claro, "
    "¿en qué casos un cliente NO puede acceder temporalmente al beneficio "
    "para sus servicio hogar Claro?"
)

respuesta = rag_chain.invoke(pregunta)

print("Pregunta:")
print(pregunta)
print("\nRespuesta generada por el modelo:")
print(respuesta.content)

Pregunta:
Según los términos y condiciones del beneficio Todo Claro, ¿en qué casos un cliente NO puede acceder temporalmente al beneficio para sus servicio hogar Claro?

Respuesta generada por el modelo:
Según los términos y condiciones de la promoción, un cliente de Todo Claro no puede acceder temporalmente al beneficio para sus servicios hogar si se detectan actividades que intentan afectar la equidad de participación. Esto incluye el uso de sistemas automáticos o semiautomáticos para obtener una ventaja injusta sobre otros participantes. Además, si se detecta falsedad en cualquier elemento de participación, se anularía cualquier entrega.

Otro caso en que un cliente podría perder temporalmente el acceso al beneficio es si se detecta que está participando con familiares cercanos, como miembros de su línea ascendente o descendente hasta el tercer grado, o integrantes de empresas del mismo grupo. Esto busca evitar conflictos de interés y asegurar la transparencia de la promoción.

Es i

### Experimento 2

In [17]:

#* Más fragmentos recuperados
#* Chunks más pequeños, menos fragmentos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, 
    add_start_index=True 
)

splits = text_splitter.split_documents(documents)

print(f"Número de fragmentos creados: {len(splits)}")
print("Ejemplo de fragmento:\n")
print(splits[0].page_content[:500], "...")


Número de fragmentos creados: 35
Ejemplo de fragmento:

BASES Y MECÁNICAS DE LA PROMOCIÓN 
 
“SU PARTICIPACIÓN EN LA PRESENTE PROMOCIÓN COMERCIAL, CONSTITUYE SU 
ADHESIÓN Y ACEPTACIÓN SIN RESERVA ALGUNA DE LOS PRESENTES TÉRMINOS, 
CONDICIONES, RESTRICCIONES Y AVISO DE PRIVACIDAD, POR LO QUE LE 
RECOMENDAMOS QUE ANTES DE PARTICIPAR LAS LEA CUIDADOSAMENTE A FIN DE QUE 
LAS ANALICE Y CONCIENTEMENTE PARTICIPE O SE ABSTENGA DE ELLO. 
 
Nombre de la promoción: “Ofertas de Hoy con Inés Básicos” 
 
Responsable y organizadora de la promoción: Desarrollo Comer ...


In [18]:

#* Directorio donde se va almacenar la base vectorial
CHROMA_DIR = BASE_DIR / "chroma_db"
persist_dir = str(CHROMA_DIR / "db_nestle_v2_langchain")

#* Creamos el vector store vacío
vector_store = Chroma(
    collection_name="nestle_v2_collection",
    embedding_function=model_embedding,
    persist_directory=persist_dir
)

#* Indexamos (añadimos) todos los fragmentos al vector store
documents_ids = vector_store.add_documents(splits)

print(f"Se indexaron {len(documents_ids)} fragmentos en Chroma.")
print(f"La base vectorial se guardó en la carpeta: {persist_dir}")

Se indexaron 35 fragmentos en Chroma.
La base vectorial se guardó en la carpeta: d:\nalvarez\100_cursos\bases_vectoriales\chroma_db\db_nestle_v2_langchain


In [19]:

#* Creación de retriever a partir del vector store
retriever_nestle_v2 = vector_store.as_retriever(
    search_kwargs={"k": 6}
)

print("Retriever creado correctamente a partir del vector store.")

Retriever creado correctamente a partir del vector store.


In [22]:
model_llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.1
)

rag_chain_v2 = (
    {
        "question": RunnablePassthrough(),
        "context": retriever_nestle_v2,
    }
    | prompt
    | model_llm
)

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [23]:
pregunta = (
    "Según los términos y condiciones del beneficio Todo Claro, "
    "¿en qué casos un cliente NO puede acceder temporalmente al beneficio "
    "para sus servicio hogar Claro?"
)

respuesta = rag_chain.invoke(pregunta)

print("Pregunta:")
print(pregunta)
print("\nRespuesta generada por el modelo:")
print(respuesta.content)

Pregunta:
Según los términos y condiciones del beneficio Todo Claro, ¿en qué casos un cliente NO puede acceder temporalmente al beneficio para sus servicio hogar Claro?

Respuesta generada por el modelo:
Según los términos y condiciones de la promoción, no se especifica en qué casos un cliente no puede acceder temporalmente al beneficio para su servicio hogar Claro. Sin embargo, el documento detalla algunas restricciones en la participación. Por ejemplo, se descalifican perfiles que intenten afectar la equidad de la participación mediante sistemas automáticos, y no pueden participar familiares cercanos que trabajen para el grupo Claro.

Además, el documento establece que el ticket participante debe tener una fecha de expedición específica (del 1 de febrero al 15 de marzo de 2019) y que solo pueden participar personas que residan en la República Mexicana con identificación oficial vigente.  También se indica que los tickets no son acumulables y solo pueden concursar una vez durante el p

### Experimento 3

In [24]:
model_llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.8
)

rag_chain_v2 = (
    {
        "question": RunnablePassthrough(),
        "context": retriever_nestle_v2,
    }
    | prompt
    | model_llm
)

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [25]:
pregunta = (
    "Según los términos y condiciones del beneficio Todo Claro, "
    "¿en qué casos un cliente NO puede acceder temporalmente al beneficio "
    "para sus servicio hogar Claro?"
)

respuesta = rag_chain.invoke(pregunta)

print("Pregunta:")
print(pregunta)
print("\nRespuesta generada por el modelo:")
print(respuesta.content)

Pregunta:
Según los términos y condiciones del beneficio Todo Claro, ¿en qué casos un cliente NO puede acceder temporalmente al beneficio para sus servicio hogar Claro?

Respuesta generada por el modelo:
Según los términos y condiciones del beneficio, un cliente no puede acceder temporalmente al beneficio para su servicio hogar Claro en los siguientes casos:

*   Si se detectan actividades que intentan afectar la equidad de participación mediante el uso de sistemas automáticos o semiautomáticos para obtener una ventaja injusta sobre otros participantes.
*   Si el participante es un familiar en línea recta ascendente o descendente en segundo grado, ni colateral hasta el tercer grado, o integrante de empresas de Grupo.
*   Si se detecta falsedad en cualquier elemento de la participación.

El documento no especifica otros casos en los que el acceso temporal al beneficio podría ser restringido.
